# MT5 Trader — Tick Data Analysis

Reusable template for exploring the tick feeds and backtesting on them.

**Sources** (all share the canonical core `[ts, symbol, bid, ask, bid_sz, ask_sz]`):
- `data/real` — real multi-provider depth dump, **May 11 – Jun 10**. Canonical best bid/ask **plus the full L2 ladder** as list columns (`bidprices/bidsizes/askprices/asksizes`). Build with `scripts/ingest_historical.py`.
- `data/ticks` — live MT5 capture, **Jun 15+** (the logger writes an extra `volume` col; `load_ticks` drops it on read).
- `data/processed` — Dukascopy placeholder (free proxy used before the real dump arrived).

`load_ticks([...], SYM)` projects every source to the core schema and merges on time, so a continuous May→now series is just a list of dirs. Read `data/real` directly when you want the depth ladder.

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

from mt5_trader.data.ingest import load_ticks
from mt5_trader.pipeline.data import build_bars, bars_per_year
from mt5_trader.backtest.engine import BTConfig, run
from mt5_trader.backtest.segment import segmented_signal, gap_bounds
from mt5_trader.strategies import REGISTRY
from mt5_trader.strategies.meanrev_pairs import spread_bars

SYMBOL = "XAUUSD"
REAL, LIVE, DUKA = "data/real", "data/ticks", "data/processed"
SOURCES = [REAL, LIVE]   # real historical + live capture
EVERY = "5m"

## 1. Coverage per source

In [ ]:
for name, src in [("real", REAL), ("live", LIVE), ("duka", DUKA)]:
    try:
        t = load_ticks(src, SYMBOL)
        print(f"{name:5} {len(t):>10,} ticks   {t['ts'].min()}  ->  {t['ts'].max()}")
    except FileNotFoundError as e:
        print(f"{name:5} -- {e}")

In [ ]:
# combined real + live, merged on time
ticks = load_ticks(SOURCES, SYMBOL)
ticks = ticks.with_columns(
    mid=(pl.col("bid") + pl.col("ask")) / 2,
    spread=pl.col("ask") - pl.col("bid"),
)
ticks = ticks.with_columns(spread_bp=1e4 * pl.col("spread") / pl.col("mid"))
print(f"{len(ticks):,} ticks  {ticks['ts'].min()} -> {ticks['ts'].max()}")
ticks.head()

## 2. Spread — distribution and intraday shape

Watch the Sunday/session-open rows: the raw multi-LP feed quotes huge spreads at the open (illiquid), which inflate the tail.

In [ ]:
print(ticks.select(
    pl.col("spread_bp").quantile(q).alias(f"p{int(q*100)}") for q in (0.5, 0.9, 0.99, 0.999)
))

by_hr = (ticks.group_by(pl.col("ts").dt.hour().alias("hour"))
         .agg(pl.col("spread_bp").median()).sort("hour"))
plt.figure(figsize=(10, 3))
plt.bar(by_hr["hour"], by_hr["spread_bp"])
plt.title(f"{SYMBOL} median spread (bp) by UTC hour")
plt.xlabel("hour"); plt.ylabel("bp"); plt.show()

In [ ]:
rate = (ticks.group_by(pl.col("ts").dt.date().alias("day"))
        .agg(pl.len().alias("ticks")).sort("day"))
plt.figure(figsize=(12, 3))
plt.bar(rate["day"].cast(str), rate["ticks"])
plt.xticks(rotation=90); plt.title("ticks per day (gaps = weekends / feed switch)"); plt.show()

## 3. Bars — price series across both feeds

In [ ]:
bars = build_bars(SYMBOL, EVERY, SOURCES)
print(f"{len(bars):,} {EVERY} bars  {bars['ts'].min()} -> {bars['ts'].max()}")
plt.figure(figsize=(12, 4))
plt.plot(bars["ts"], bars["close"])
plt.title(f"{SYMBOL} {EVERY} close (real + live)"); plt.show()
bars.tail()

## 4. Depth ladder & providers

`data/real` keeps the full L2 ladder as list columns alongside the canonical best bid/ask — `load_ticks` projects to best-only, so read the parquet directly for depth. `provider` is dropped at ingest; read the raw dump for the per-provider breakdown.

In [ ]:
# depth ladder straight from data/real (kept by ingest_historical)
ladder = (pl.scan_parquet(f"data/real/{SYMBOL}/*.parquet")
          .select("ts", "bid", "ask", "bidprices", "bidsizes", "askprices", "asksizes")
          .head(3).collect())
print(ladder)
print("levels in first row:", len(ladder["bidprices"][0]))

# per-provider quote counts (raw dump only — provider isn't in canonical)
RAW = "/Volumes/SK Storage-1/Trading/2026-05-11_2026-06-10"
raw = pl.scan_parquet(f"{RAW}/{SYMBOL}_2026_05_12.parquet").select("provider").head(20000).collect()
raw.group_by("provider").len().sort("len", descending=True)

## 5. Backtest on the combined data

Same engine as `scripts/run_backtest.py`; just point `build_bars` at the source list. Swap `vwap_trend` for any key in `REGISTRY`, or build a spread for `ratio_mr`.

In [ ]:
bars = build_bars(SYMBOL, EVERY, SOURCES)
btc = BTConfig(slippage_bps=0.5, bars_per_year=bars_per_year(EVERY))
print(f"{len(gap_bounds(bars))} contiguous segment(s) — multi-day data holes split out\n")

# segmented_signal: signal per segment so a stitched-feed gap can't whipsaw a
# rolling indicator (no-op on one continuous feed).
for strat_name in ("vwap_trend", "london_orb", "sweep_fade"):
    strat = REGISTRY[strat_name]()
    res = run(bars, segmented_signal(bars, strat), btc)
    m = res.metrics
    print(f"{strat_name:12} ret {m['total_return']:+.4f}  sharpe {m['sharpe']:6.2f}  maxDD {m['max_drawdown']:.4f}")

In [ ]:
# ratio_mr: signal off the XAU/XAG spread, deployed params (z_n=288, regime gate on)
from mt5_trader.strategies.meanrev_pairs import RatioMeanRev
xau = build_bars("XAUUSD", EVERY, SOURCES)
xag = build_bars("XAGUSD", EVERY, SOURCES)
spread = spread_bars(xau, xag, beta=1.0)
strat = RatioMeanRev(z_n=288, regime_filter=True)
res = run(spread, segmented_signal(spread, strat),
          BTConfig(slippage_bps=0.5, bars_per_year=bars_per_year(EVERY)))
res.metrics